# 14 實戰案例：松柏護理之家退伍軍人症群聚調查報告

整合全書技能，從原始資料到完整疫調報告。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

plt.rcParams["font.size"] = 12
pd.set_option("display.max_columns", 40)

---
## 1. 背景與通報

2026 年 1 月中旬，衛生局接獲松柏護理之家通報：
近日多名住民出現肺炎症狀，疑似退伍軍人症（Legionnaires' disease）群聚事件。

- **機構**：松柏護理之家（280 位住民）
- **住民特性**：60–98 歲長者，多數有慢性病
- **設施**：3 層樓 × 2 翼區（A/B），設有淋浴間與水療池
- **通報日期**：2026 年 1 月

In [ ]:
# --- Step 1: 讀取資料 ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

date_cols = ["symptom_onset_date", "notification_date", "hospitalization_date",
             "death_date", "facility_admission_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 衍生變項
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = df["clinical_severity"].isin(["severe"]).astype(int)
df["age_group"] = pd.cut(df["age"], bins=[59, 69, 79, 89, 100],
                         labels=["60-69", "70-79", "80-89", "90+"])

cases = df[df["infected"] == 1].copy()

print(f"資料筆數: {len(df)} 位住民")
print(f"欄位數: {df.shape[1]} 欄")
print(f"感染者: {len(cases)} 人")
print(f"日期範圍: {cases['symptom_onset_date'].min().date()} ~ {cases['symptom_onset_date'].max().date()}")

---
## 2. 方法

- **個案定義**：住民於調查期間出現肺炎症狀（`clinical_severity != 'not_ill'`）
- **確定病例**：經實驗室確認之退伍軍人桿菌感染（`lab_confirmed == 1`）
- **資料收集**：回溯式調查，收集人口學、暴露史、臨床資料共 32 欄

---
## 3. 描述性流行病學

In [ ]:
# --- 3a. 疫情摘要 ---
n_total = len(df)
n_infected = int(df["infected"].sum())
n_confirmed = int(df["lab_confirmed"].sum())
n_hospitalized = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "總住民數": n_total,
    "感染人數": n_infected,
    "實驗室確認": n_confirmed,
    "住院人數": n_hospitalized,
    "ICU 人數": n_icu,
    "死亡人數": n_deaths,
    "侵襲率 (AR)": f"{n_infected / n_total:.1%}",
    "致死率 (CFR)": f"{n_deaths / n_infected:.1%}",
}

print("=" * 40)
print("  松柏護理之家退伍軍人症群聚事件摘要")
print("=" * 40)
for k, v in summary.items():
    print(f"  {k}: {v}")
print("=" * 40)

In [ ]:
# --- 3b. 流行曲線 (Epidemic Curve) ---
import matplotlib.dates as mdates

daily = cases.groupby("symptom_onset_date").size()
# 補齊完整日期範圍（含爆發前 3 天背景期）
full_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily = daily.reindex(full_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#2c7fb8", edgecolor="white", linewidth=0.5)
ax.set_title("松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
             fontsize=13, fontweight="bold")
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)

ax.set_xlim(daily.index.min() - pd.Timedelta(hours=12),
            daily.index.max() + pd.Timedelta(hours=12))
ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# 標註流行高峰
peak_date = daily.idxmax()
ax.annotate(f"高峰: {peak_date.strftime('%m/%d')}\n({daily.max()} 例)",
            xy=(peak_date, daily.max()), xytext=(15, 5),
            textcoords="offset points", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="red"))

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"→ 流行曲線呈共同暴露源型態（common source）")
print(f"→ 發病高峰: {peak_date.date()}，共 {daily.max()} 例")

In [ ]:
# --- 3c. 人、地分布 ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 年齡分布
for label, grp in df.groupby("infected"):
    tag = "感染" if label == 1 else "未感染"
    axes[0].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[0].set_xlabel("年齡")
axes[0].set_ylabel("人數")
axes[0].set_title("年齡分布")
axes[0].legend()

# 性別 × 感染
sex_ct = df.groupby("sex")["infected"].agg(["sum", "count"])
sex_ct["ar"] = sex_ct["sum"] / sex_ct["count"] * 100
axes[1].bar(sex_ct.index, sex_ct["ar"], color=["#4C72B0", "#DD8452"])
axes[1].set_ylabel("侵襲率 (%)")
axes[1].set_title("性別侵襲率")
for i, (idx, row) in enumerate(sex_ct.iterrows()):
    axes[1].text(i, row["ar"] + 1, f"{row['ar']:.1f}%", ha="center")

# 樓層翼區侵襲率
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].set_ylabel("侵襲率 (%)")
axes[2].set_title("樓層翼區侵襲率")
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"→ 2F-A ({zone[zone['label']=='2F-A']['ar'].values[0]:.1f}%) 和 3F-B ({zone[zone['label']=='3F-B']['ar'].values[0]:.1f}%) 侵襲率最高")

---
## 4. 分析性流行病學

In [ ]:
# --- 4a. 淋浴暴露 2×2 表 ---
ct = pd.crosstab(df["shower_use"], df["infected"], margins=True)
ct.index = ["未淋浴", "淋浴", "合計"]
ct.columns = ["未感染", "感染", "合計"]
print("=== 淋浴使用 × 感染狀態 ===")
print(ct)

# 風險比 (RR)
a, b = 76, 148  # 淋浴且感染, 淋浴總人數
c, d = 45, 132  # 未淋浴且感染, 未淋浴總人數
a = int(df[(df["shower_use"] == 1) & (df["infected"] == 1)].shape[0])
b = int(df[df["shower_use"] == 1].shape[0])
c = int(df[(df["shower_use"] == 0) & (df["infected"] == 1)].shape[0])
d = int(df[df["shower_use"] == 0].shape[0])

rr = (a / b) / (c / d)
print(f"\n淋浴組侵襲率: {a}/{b} = {a/b:.1%}")
print(f"非淋浴組侵襲率: {c}/{d} = {c/d:.1%}")
print(f"風險比 (RR): {rr:.2f}")

# 卡方檢定
chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df["shower_use"], df["infected"]))
print(f"卡方檢定: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"\n→ 淋浴使用與感染有統計顯著關聯 (p < 0.05)" if p < 0.05 else "")

In [ ]:
# --- 4b. 分層分析：functional_status 是否為干擾因子？ ---
print("=== 依 functional_status 分層的淋浴侵襲率 ===")
rows = []
for fs, grp in df.groupby("functional_status"):
    for su in [1, 0]:
        sub = grp[grp["shower_use"] == su]
        n = len(sub)
        cases_n = int(sub["infected"].sum())
        ar = cases_n / n * 100 if n > 0 else 0
        rows.append({"functional_status": fs, "shower_use": su,
                     "n": n, "cases": cases_n, "ar": f"{ar:.1f}%"})

strat_df = pd.DataFrame(rows)
print(strat_df.to_string(index=False))

print("\n→ 臥床住民幾乎不淋浴，且感染率較低")
print("→ functional_status 是干擾因子：影響淋浴能力也影響暴露機會")
print("→ 需要用邏輯斯迴歸做多變項調整")

In [ ]:
# --- 4c. 邏輯斯迴歸：adjusted OR ---
import statsmodels.api as sm

model_df = df[["infected", "shower_use", "age", "sex",
               "comorbidity_chf", "comorbidity_dm", "comorbidity_copd",
               "immunosuppressed", "hydrotherapy_use"]].copy()
model_df["sex_male"] = (model_df["sex"] == "M").astype(int)
model_df = model_df.drop(columns=["sex"])

X = model_df.drop(columns=["infected"])
X = sm.add_constant(X)
y = model_df["infected"]

logit = sm.Logit(y, X).fit(disp=0)

# OR table
or_df = pd.DataFrame({
    "OR": np.exp(logit.params),
    "95% CI lower": np.exp(logit.conf_int()[0]),
    "95% CI upper": np.exp(logit.conf_int()[1]),
    "p-value": logit.pvalues,
}).drop(index="const")

or_df["OR (95% CI)"] = or_df.apply(
    lambda r: f"{r['OR']:.2f} ({r['95% CI lower']:.2f}–{r['95% CI upper']:.2f})", axis=1
)

print("=== 多變項邏輯斯迴歸結果 ===")
print(or_df[["OR (95% CI)", "p-value"]].to_string())
print("\n→ 調整年齡、性別、共病後，淋浴使用的 adjusted OR 及其顯著性")

---
## 5. 時間空間分析

In [ ]:
# --- 5a. 各樓層翼區流行曲線比較 ---
cases["zone"] = cases["floor"].astype(str) + "F-" + cases["wing"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
axes = axes.flatten()

for i, (zone_name, grp) in enumerate(cases.groupby("zone")):
    daily_z = grp.groupby("symptom_onset_date").size()
    daily_z = daily_z.reindex(full_range, fill_value=0)
    axes[i].bar(daily_z.index, daily_z.values, width=1.0,
                color="#2c7fb8", edgecolor="white", linewidth=0.5)
    n_cases = len(grp)
    n_total_zone = len(df[(df["floor"].astype(str) + "F-" + df["wing"]) == zone_name])
    ar = n_cases / n_total_zone * 100
    axes[i].set_title(f"{zone_name}  (AR={ar:.0f}%, n={n_cases})")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(False)
    axes[i].spines["top"].set_visible(False)
    axes[i].spines["right"].set_visible(False)
    axes[i].yaxis.set_major_locator(plt.MaxNLocator(integer=True))

fig.suptitle("松柏護理之家各區域退伍軍人症流行曲線，依發病日，2026 年 1 月",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("→ 2F 和 3F-B 區的流行曲線最為密集")
print("→ 各區發病時間相近，支持共同暴露源假說")

In [ ]:
# --- 5b. 空間熱力圖：侵襲率 × 致死率 ---
zone_stats = df.groupby(["floor", "wing"]).agg(
    n=("infected", "count"),
    cases=("infected", "sum"),
    deaths=("outcome", lambda x: (x == "dead").sum()),
).reset_index()
zone_stats["ar"] = zone_stats["cases"] / zone_stats["n"] * 100
zone_stats["cfr"] = np.where(
    zone_stats["cases"] > 0,
    zone_stats["deaths"] / zone_stats["cases"] * 100,
    0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for idx, (metric, title, cmap) in enumerate([
    ("ar", "侵襲率 (%)", "YlOrRd"),
    ("cfr", "致死率 (%)", "YlOrRd"),
]):
    pivot = zone_stats.pivot(index="floor", columns="wing", values=metric)
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap=cmap, ax=axes[idx],
                cbar_kws={"label": "%"})
    axes[idx].set_title(title)
    axes[idx].set_ylabel("樓層")
    axes[idx].set_xlabel("翼區")

plt.suptitle("空間分布：侵襲率與致死率", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. 進階分析

In [ ]:
# --- 6a. Kaplan-Meier 存活曲線 ---
from lifelines import KaplanMeierFitter

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["event"] = (cases["outcome"] == "dead").astype(int)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days
cases = cases[cases["time_to_event"] > 0]

fig, ax = plt.subplots(figsize=(8, 5))
kmf = KaplanMeierFitter()

for sev in ["severe", "moderate", "mild"]:
    mask = cases["clinical_severity"] == sev
    if mask.sum() > 0:
        kmf.fit(cases.loc[mask, "time_to_event"],
                cases.loc[mask, "event"], label=sev)
        kmf.plot_survival_function(ax=ax)

ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_title("Kaplan-Meier 存活曲線（依嚴重度分層）")
plt.tight_layout()
plt.show()

print("→ 重症個案存活率明顯低於中度和輕症")

In [ ]:
# --- 6b. 機器學習：危險因子重要性排序 ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = ["floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
            "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat_cols),
    ("bin", "passthrough", bin_cols),
])

pipe = Pipeline([
    ("pre", preprocess),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
])

X = df[num_cols + cat_cols + bin_cols]
y = df["infected"]

auc_scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print(f"Random Forest 5-fold CV AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")

# Feature importance
pipe.fit(X, y)
X_transformed = pipe.named_steps["pre"].transform(X)
feat_names = pipe.named_steps["pre"].get_feature_names_out()

perm = permutation_importance(pipe.named_steps["rf"], X_transformed, y,
                               n_repeats=10, random_state=42)

imp_df = pd.DataFrame({
    "feature": feat_names,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=True).tail(8)

# 將機器產生的特徵名稱對應為人可閱讀的中文標籤
label_map = {
    "num__age": "年齡",
    "cat__sex_M": "性別（男）",
    "cat__smoking_history_former": "吸菸史（曾吸菸）",
    "cat__smoking_history_current": "吸菸史（目前吸菸）",
    "cat__functional_status_wheelchair": "功能狀態（輪椅）",
    "cat__functional_status_ambulatory": "功能狀態（可行走）",
    "cat__wing_B": "翼區（B 區）",
    "bin__floor": "樓層",
    "bin__comorbidity_chf": "共病：心衰竭",
    "bin__comorbidity_dm": "共病：糖尿病",
    "bin__comorbidity_cancer": "共病：癌症",
    "bin__comorbidity_copd": "共病：COPD",
    "bin__immunosuppressed": "免疫抑制",
    "bin__shower_use": "淋浴使用",
    "bin__hydrotherapy_use": "水療使用",
}
imp_df["label"] = imp_df["feature"].map(label_map).fillna(imp_df["feature"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["label"], imp_df["importance"], color="steelblue")
ax.set_xlabel("Permutation Importance")
ax.set_title("感染預測：前 8 大重要特徵")
plt.tight_layout()
plt.show()

---
## 7. 討論

### 感染源研判

綜合以上分析結果：

| 證據 | 發現 | 指向 |
|------|------|------|
| 流行曲線 | 共同暴露源型態 | 持續性環境暴露 |
| 空間分布 | 2F 和 3F-B 侵襲率 > 50% | 特定樓層水管系統 |
| 暴露分析 | 淋浴 RR > 1，調整後仍顯著 | 淋浴水為傳播途徑 |
| 分層分析 | 臥床者（不淋浴）感染率低 | 排除空氣傳播為主因 |

**結論：淋浴供水系統為最可能的感染源。**

### 建議介入措施

1. **立即**：停用 2F 及 3F-B 淋浴設施
2. **短期**：全棟水系統加熱消毒（> 70°C）
3. **中期**：更換老舊水管、加裝水溫控制
4. **長期**：建立水質定期監測制度

In [ ]:
# --- 8. 結論：結案報告摘要表 ---
report = pd.DataFrame([
    ["事件類型", "退伍軍人症群聚"],
    ["機構", "松柏護理之家"],
    ["調查期間", f"{cases['symptom_onset_date'].min().date()} ~ {cases['symptom_onset_date'].max().date()}"],
    ["總住民", f"{n_total} 人"],
    ["感染人數", f"{n_infected} 人 (AR {n_infected/n_total:.1%})"],
    ["死亡人數", f"{n_deaths} 人 (CFR {n_deaths/n_infected:.1%})"],
    ["住院", f"{n_hospitalized} 人"],
    ["ICU", f"{n_icu} 人"],
    ["最高風險區", "2F-A (54.5%), 3F-B (57.4%)"],
    ["主要危險因子", "淋浴使用"],
    ["推定感染源", "淋浴供水系統"],
    ["建議措施", "停用淋浴 → 加熱消毒 → 更換水管 → 定期監測"],
], columns=["項目", "內容"])

print("=" * 50)
print("    松柏護理之家退伍軍人症群聚事件 — 結案報告")
print("=" * 50)
for _, row in report.iterrows():
    print(f"  {row['項目']:　<10}{row['內容']}")
print("=" * 50)
print("\n→ 本報告由 Python 自動產出，所有分析步驟均可重現")
print("→ 完整程式碼見本 notebook")

---
## 把圖表輸出成 PPTX / DOCX 報告

分析做完、結論也寫好了，但長官要看的通常不是 notebook，而是一份可以直接上呈的簡報或 Word 報告。與其把圖表截圖、手動貼進 PowerPoint／Word——那種做法每次資料更新都要重做一遍，還很容易貼錯版本——不如用 `python-pptx` / `python-docx` 直接把前面算好的圖表和數字**程式化組成報告**：完全不需要手動複製貼上，而且**可重現**（呼應 Ch13：只要重新執行這個 notebook，報告就會用最新資料重新產生一次，結果永遠跟分析同步）。

下面用同一張流行曲線和一組摘要數字，同時輸出一份 `.pptx` 簡報和一份 `.docx` 報告，示範「程式化產報告」這個技能本身。

下面這個 cell 準備所有匯出共用的素材：把圖存成記憶體裡的 PNG（不寫進硬碟）、整理摘要數字、建立一個暫存輸出資料夾。這裡刻意重新彙總一次每日病例數，讓這個 cell 不依賴前面 cell 的執行順序也能獨立重跑。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `def fig_to_png_bytes(fig)` | 把 matplotlib figure 存成記憶體裡的 PNG（`io.BytesIO`），不落地成檔案就能直接塞進報告 |
> | `daily_for_export = cases.groupby(...)` | 重新彙總每日病例數，避免依賴前面 cell 是否被覆寫，讓這個 cell 能獨立重跑 |
> | `fig_epi_curve, ax_epi_curve = plt.subplots(...)` | 重畫一張乾淨的流行曲線——3b 用過的 `fig` 變數，早已被後續 cell（6a、6b…）覆寫成別的圖 |
> | `curve_png = fig_to_png_bytes(fig_epi_curve)` | 把流行曲線轉成記憶體 PNG，PPTX 和 DOCX 會共用同一份 |
> | `summary_rows = [...]` | 沿用第 3 節已經算好的 `n_total` / `n_infected` / `n_deaths`，組成報告要用的摘要列 |
> | `outdir = tempfile.mkdtemp(prefix="sitrep_")` | 建立系統暫存資料夾——輸出檔案不寫進專案目錄，不會被 git 追蹤到 |

> ⚠️ **`BytesIO` 只能讀一次**：圖片存進 `curve_png` 之後，第一次 `add_picture()` 讀完游標就停在檔案尾端；PPTX、DOCX 都要用同一張圖時，第二次讀之前必須先 `curve_png.seek(0)` 把游標倒回開頭，否則會插入一張空白圖。

In [ ]:
# --- 把共用素材準備好：圖表轉成記憶體 PNG、整理摘要數字、開暫存資料夾 ---
import io
import os
import tempfile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
from docx import Document
from docx.shared import Inches as DInches


def fig_to_png_bytes(fig):
    """把 matplotlib figure 存成記憶體裡的 PNG——不落地成檔案，直接塞進報告。"""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf


# 重新彙總每日病例數，讓這個 cell 不依賴前面 cell 的執行順序，可以獨立重跑
daily_for_export = cases.groupby("symptom_onset_date").size()
full_range_for_export = pd.date_range(
    daily_for_export.index.min() - pd.Timedelta(days=3),
    daily_for_export.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily_for_export = daily_for_export.reindex(full_range_for_export, fill_value=0)

fig_epi_curve, ax_epi_curve = plt.subplots(figsize=(9, 4))
ax_epi_curve.bar(daily_for_export.index, daily_for_export.values, width=1.0,
                  color="#2c7fb8", edgecolor="white", linewidth=0.5)
ax_epi_curve.set_title("松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
                        fontsize=12, fontweight="bold")
ax_epi_curve.set_xlabel("發病日期（Date of Symptom Onset）")
ax_epi_curve.set_ylabel("病例數（Number of Cases）")
ax_epi_curve.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax_epi_curve.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig_epi_curve.autofmt_xdate(rotation=45)
ax_epi_curve.set_ylim(bottom=0)
ax_epi_curve.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax_epi_curve.grid(False)
ax_epi_curve.spines["top"].set_visible(False)
ax_epi_curve.spines["right"].set_visible(False)
plt.tight_layout()

curve_png = fig_to_png_bytes(fig_epi_curve)   # <- reuse the epidemic-curve figure

summary_rows = [
    ("住民數", f"{n_total}"),
    ("感染", f"{n_infected}（{n_infected / n_total:.1%}）"),
    ("死亡", f"{n_deaths}（{n_deaths / n_infected:.1%}）"),
]

outdir = tempfile.mkdtemp(prefix="sitrep_")    # 寫到暫存資料夾，不污染專案
print(f"暫存輸出資料夾：{outdir}")

接著把流行曲線和摘要數字組成一份簡報：一張標題頁 + 一張內容頁（圖表 + 表格）。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `prs.slides.add_slide(prs.slide_layouts[0])` | 版面 0 = 標題投影片，內建主標題與副標題 placeholder |
> | `slide1.placeholders[1].text = ...` | 副標題 placeholder（index 1），放關鍵指標讓人一眼看到侵襲率、致死率 |
> | `prs.slides.add_slide(prs.slide_layouts[5])` | 版面 5 = 空白投影片，自己用 textbox／picture／table 排版 |
> | `curve_png.seek(0)` | 插入圖片前重設 `BytesIO` 游標（上一個 cell 提過的雷） |
> | `slide2.shapes.add_picture(curve_png, ...)` | 把記憶體裡的 PNG 直接插入投影片，不需要先存成檔案 |
> | `slide2.shapes.add_table(rows, cols, ...).table` | 在投影片上畫一個表格，把 `summary_rows` 逐列填入 |
> | `prs.save(pptx_path)` | 存到暫存資料夾，檔名固定方便下次重跑時直接覆蓋 |

In [ ]:
# --- 匯出 PPTX 簡報 ---
prs = Presentation()

# 投影片 1：標題頁（版面 0 = 標題投影片，含主標題 + 副標題 placeholder）
slide1 = prs.slides.add_slide(prs.slide_layouts[0])
slide1.shapes.title.text = "松柏護理之家退伍軍人症群聚 SitRep"
slide1.placeholders[1].text = (
    f"侵襲率 {n_infected / n_total:.1%}｜致死率 {n_deaths / n_infected:.1%}"
)

# 投影片 2：流行曲線 + 摘要表格（版面 5 = 空白版面）
slide2 = prs.slides.add_slide(prs.slide_layouts[5])
title_box = slide2.shapes.add_textbox(Inches(0.4), Inches(0.3), Inches(9), Inches(0.6))
title_box.text_frame.text = "流行曲線與摘要指標"

curve_png.seek(0)   # BytesIO 只能讀一次，插入圖片前一定要重設游標
slide2.shapes.add_picture(curve_png, Inches(0.4), Inches(1.1), width=Inches(6))

rows_n, cols_n = len(summary_rows) + 1, 2
tbl = slide2.shapes.add_table(
    rows_n, cols_n, Inches(6.8), Inches(1.1), Inches(2.7), Inches(1.6)
).table
tbl.cell(0, 0).text = "項目"
tbl.cell(0, 1).text = "數值"
for i, (label, value) in enumerate(summary_rows, start=1):
    tbl.cell(i, 0).text = label
    tbl.cell(i, 1).text = value

pptx_path = os.path.join(outdir, "legionella_sitrep.pptx")
prs.save(pptx_path)

同一套素材，再組一份 Word 報告——比起簡報，Word 更適合列印存檔或附加在正式公文、email 裡。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `doc.add_heading(..., level=0)` | Word 的「標題」樣式，作為文件主標題 |
> | `doc.add_heading("摘要", level=1)` | 「標題 1」樣式，作為小節標題 |
> | `doc.add_paragraph(...)` | 用一段文字寫出關鍵發現，長官不用自己爬 notebook 找數字 |
> | `curve_png.seek(0)` | 同一個 `BytesIO` 在 PPTX 那格已經讀過一次，這裡要再 `seek(0)` 才能重新讀取 |
> | `doc.add_picture(curve_png, width=DInches(5.5))` | 插入流行曲線；`DInches` 是 `docx.shared.Inches` 改的別名，避免跟 pptx 的 `Inches` 撞名 |
> | `doc.add_table(rows=1, cols=2, style="Light Grid Accent 1")` | 建一個套用內建樣式的表格，先放表頭 |
> | `table.add_row()` | 每個 `summary_rows` 項目動態加一列，不用事先算好列數 |
> | `doc.save(docx_path)` | 存到跟 PPTX 同一個暫存資料夾 |

In [ ]:
# --- 匯出 DOCX 報告 ---
doc = Document()
doc.add_heading("松柏護理之家退伍軍人症群聚調查報告", level=0)

doc.add_heading("摘要", level=1)
doc.add_paragraph(
    f"本次群聚事件共 {n_total} 位住民，{n_infected} 人感染"
    f"（侵襲率 {n_infected / n_total:.1%}），{n_deaths} 人死亡"
    f"（致死率 {n_deaths / n_infected:.1%}）。流行曲線呈共同暴露源型態，"
    "推定感染源為淋浴供水系統（詳見第 7 節討論）。"
)

doc.add_heading("流行曲線", level=1)
curve_png.seek(0)   # 同一份 BytesIO 在 PPTX 那格已讀過一次，這裡要重新 seek(0)
doc.add_picture(curve_png, width=DInches(5.5))

doc.add_heading("關鍵數字", level=1)
table = doc.add_table(rows=1, cols=2, style="Light Grid Accent 1")
table.rows[0].cells[0].text = "項目"
table.rows[0].cells[1].text = "數值"
for label, value in summary_rows:
    row = table.add_row()
    row.cells[0].text = label
    row.cells[1].text = value

docx_path = os.path.join(outdir, "legionella_sitrep.docx")
doc.save(docx_path)

最後印出兩個檔案的實際路徑和大小，確認報告真的落地、不是 0 bytes 的空殼檔案。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `os.path.getsize(path)` | 讀取檔案實際位元組數，用來驗證檔案有實際內容 |

> 🎁 **洞見**：程式化產報告 = 一鍵重跑、格式一致、零手工錯誤。資料更新、結論修正，重新執行這個 notebook 就能重新產生一致格式的簡報和 Word 報告——不用再手動截圖貼投影片，也不會有「這份報告用的是哪個版本的圖」這種問題。

In [ ]:
# --- 驗證：確認兩份報告真的落地、且不是空殼檔案 ---
for path in (pptx_path, docx_path):
    size_kb = os.path.getsize(path) / 1024
    print(f"{path}  ({size_kb:.1f} KB)")